# RentalEval — Agent Experiments

Use this notebook to iterate on agent prompts before wiring them into the FastAPI backend.

**Pre-requisites:**
1. Copy `backend/.env.example` → `backend/.env` and fill in `ANTHROPIC_API_KEY` (+ optionally `TAVILY_API_KEY`)
2. Run from the `backend/` directory so imports resolve:
   ```bash
   cd backend
   uv run jupyter notebook notebooks/agent_experiments.ipynb
   ```

## 0. Setup

In [ ]:
import sys, os
# Make sure we can import from backend/app/
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
TAVILY_API_KEY    = os.environ.get('TAVILY_API_KEY', '')

assert ANTHROPIC_API_KEY, 'Set ANTHROPIC_API_KEY in backend/.env'
print('Keys loaded OK')

In [ ]:
# Test address + realistic user profile — edit these to try different scenarios
TEST_ADDRESS = '1600 Amphitheatre Pkwy, Mountain View, CA'
TEST_ZIP     = '94043'

TEST_PROFILE = {
    'name': 'Alex',
    'household': 'solo',
    'ethnicity': 'South Asian',
    'transportation': 'no_car',
    'work_location': '500 Terry Francois Blvd, San Francisco, CA',
    'work_schedule': '9-5',
    'exercise_routine': 'running',
    'food_preferences': ['vegetarian', 'indian', 'cafes'],
    'has_pets': False,
    'budget': 3200,
    'priorities': ['transportation', 'food', 'safety', 'lifestyle',
                   'convenience', 'utilities', 'building', 'future_risk'],
}

## 1. How agents work: LangGraph ReAct

Each subagent is a **ReAct loop** (Reason → Act → Observe → repeat):
```
System prompt (your role + output format)
    ↓
User message (address + profile)
    ↓
Claude reasons → decides which tool to call
    ↓
Tool executes (Tavily search, Yelp, Maps, etc.)
    ↓
Claude observes result → reasons again → another tool OR final answer
    ↓
Final JSON output: { score, summary, details }
```

`create_react_agent(llm, tools, prompt=system_prompt)` from LangGraph wires this up automatically.
The agent loop terminates when Claude outputs a message with no tool calls.

## 2. Build a minimal agent from scratch (no app imports)

Start here to understand the primitives before using our app code.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# --- Minimal stub tool ---
@tool
def web_search(query: str) -> str:
    """Search the web for current information about a location or topic."""
    if not TAVILY_API_KEY:
        return f'[web_search stub] No TAVILY_API_KEY — pretend you found useful info for: {query}'
    from tavily import TavilyClient
    client = TavilyClient(api_key=TAVILY_API_KEY)
    results = client.search(query, max_results=3)
    return '\n\n'.join(r['content'] for r in results.get('results', []))

# --- LLM ---
llm = ChatAnthropic(model='claude-sonnet-4-6', api_key=ANTHROPIC_API_KEY, max_tokens=2048)

# --- System prompt (this is what we iterate on) ---
SYSTEM_PROMPT = """
You are the Safety Agent for a rental property evaluator.

Evaluate: crime risk, natural disaster risk (flood/quake/wildfire), air quality, and environmental hazards.
Use web_search to gather real data. Prioritize findings relevant to the user's profile.

Respond ONLY with a JSON object:
{"score": <0-100>, "summary": "<one sentence>", "details": "<multi-paragraph findings>"}
"""

agent = create_react_agent(llm, [web_search], prompt=SYSTEM_PROMPT)
print('Agent built OK')

In [ ]:
import asyncio, json, re

user_msg = f"""
Evaluate safety for: {TEST_ADDRESS} (zip: {TEST_ZIP})
User profile: {TEST_PROFILE}
"""

result = await agent.ainvoke({'messages': [('user', user_msg)]})

# Print full message trace so you can see the tool calls
for msg in result['messages']:
    role = type(msg).__name__
    content = msg.content if isinstance(msg.content, str) else str(msg.content)[:200]
    print(f'[{role}]\n{content[:400]}\n')

In [ ]:
# Parse final JSON output
final_content = ''
for msg in reversed(result['messages']):
    if hasattr(msg, 'content') and isinstance(msg.content, str) and msg.content.strip():
        final_content = msg.content.strip()
        break

json_match = re.search(r'\{.*\}', final_content, re.DOTALL)
if json_match:
    parsed = json.loads(json_match.group())
    print(f"Score:   {parsed['score']}/100")
    print(f"Summary: {parsed['summary']}")
    print(f"\nDetails:\n{parsed['details']}")
else:
    print('Could not parse JSON. Raw output:')
    print(final_content)

## 3. Test each agent using app code

Once the basic loop looks good, run the actual agent modules.

In [ ]:
# Run one agent at a time to test + refine its prompt

# Change this import to test any agent:
# from app.agents.transportation import run_transportation_agent as run_agent_fn
# from app.agents.food import run_food_agent as run_agent_fn
# from app.agents.lifestyle import run_lifestyle_agent as run_agent_fn
# from app.agents.convenience import run_convenience_agent as run_agent_fn
# from app.agents.utilities import run_utilities_agent as run_agent_fn
# from app.agents.building import run_building_agent as run_agent_fn
# from app.agents.future_risk import run_future_risk_agent as run_agent_fn

from app.agents.safety import run_safety_agent as run_agent_fn

output = await run_agent_fn(TEST_ADDRESS, TEST_ZIP, TEST_PROFILE, ANTHROPIC_API_KEY)

print(f"Score:   {output['score']}/100")
print(f"Summary: {output['summary']}")
print(f"\nDetails:\n{output['details']}")

## 4. Prompt iteration

Edit the `IMPROVED_PROMPT` below, run the cell, compare outputs.

In [ ]:
from app.agents.base import build_agent, run_agent
from app.tools.search import web_search as ws_tool
from app.tools.air_quality import get_air_quality

# ── Iterate here ──────────────────────────────────────────────────────────────
IMPROVED_PROMPT = """
You are the Safety Agent for a rental property evaluator.

Your role: give a DATA-DRIVEN safety evaluation that is personalized to this specific user.

Research the following for the given address — use web_search and get_air_quality:
1. Overall crime rate vs city average (violent + property crime index)
2. Specific crime types relevant to the user's schedule (e.g. night-shift workers → late-night crimes)
3. Natural disaster risk: FEMA flood zone, earthquake risk, wildfire risk (if California)
4. Environmental: Superfund sites within 1 mile, industrial facilities
5. Air Quality Index (AQI) — note impact on anyone with respiratory concerns
6. Estimated renter's insurance cost for the zip code

Personalization rules:
- If user has pets → note off-leash parks and pet safety in the area
- If user exercises outdoors → note safety for early morning/evening runs
- If night-shift → weight late-night crime data more heavily
- If family/kids → weight sex offender proximity and school safety

Score guide:
- 80-100: Very safe, minimal risk across all categories
- 60-79:  Generally safe, some elevated risk in 1-2 categories
- 40-59:  Mixed — significant risk in at least one category
- 20-39:  Concerning — multiple elevated risk factors
- 0-19:   Avoid — serious safety concerns

Output ONLY valid JSON:
{"score": <integer 0-100>, "summary": "<one sentence>", "details": "<3-5 paragraphs>"}
"""
# ──────────────────────────────────────────────────────────────────────────────

agent_v2 = build_agent(IMPROVED_PROMPT, [ws_tool, get_air_quality], ANTHROPIC_API_KEY)
output_v2 = await run_agent(agent_v2, f'Evaluate safety for: {TEST_ADDRESS} (zip: {TEST_ZIP})\nProfile: {TEST_PROFILE}')

print(f"Score:   {output_v2['score']}/100")
print(f"Summary: {output_v2['summary']}")
print(f"\nDetails:\n{output_v2['details']}")

## 5. Run the full orchestrator (all 8 agents in parallel)

This is the closest simulation of what the API endpoint does.

In [ ]:
from app.agents.orchestrator import run_evaluation
import uuid

evaluation_id = str(uuid.uuid4())
full_report = {}

print(f'Starting evaluation for: {TEST_ADDRESS}\n')

async for event_type, payload in run_evaluation(
    address=TEST_ADDRESS,
    zip_code=TEST_ZIP,
    profile=TEST_PROFILE,
    api_key=ANTHROPIC_API_KEY,
    evaluation_id=evaluation_id,
):
    if event_type == 'agent_update':
        agent = payload['agent']
        status = payload['status']
        if status == 'running':
            print(f'  → {agent:<20} running...')
        elif status == 'complete':
            score = payload.get('score', '?')
            summary = payload.get('summary', '')
            print(f'  ✓ {agent:<20} {score:>3}/100  {summary[:60]}')
    elif event_type == 'complete':
        full_report = payload['report']
        print(f"\n{'='*60}")
        print(f"OVERALL SCORE: {full_report['overall_score']}/100")
        print(f"{'='*60}")

In [ ]:
# Full report structure
import json

print('=== PERSONA NARRATIVE ===')
print(full_report.get('persona_narrative', 'N/A'))

print('\n=== PROS ===')
for p in full_report.get('pros', []):
    print(f'  + {p}')

print('\n=== CONS ===')
for c in full_report.get('cons', []):
    print(f'  - {c}')

print('\n=== RED FLAGS ===')
for r in full_report.get('red_flags', []):
    print(f'  ⚠ {r}')

print('\n=== MONTHLY COST ESTIMATE ===')
print(json.dumps(full_report.get('monthly_cost_estimate', {}), indent=2))

print('\n=== SECTION SCORES ===')
for dim, data in full_report.get('sections', {}).items():
    print(f'  {dim:<20} {data["score"]:>3}/100  {data["summary"][:60]}')

## 6. Extended thinking (optional — Claude claude-opus-4-6 only)

For deeper reasoning on complex agents (e.g. future_risk), you can enable extended thinking.
This costs more but produces more thorough analysis.

In [ ]:
# Extended thinking — uses claude-opus-4-6, costs more tokens
from langchain_anthropic import ChatAnthropic
from langgraph.prebuilt import create_react_agent

llm_thinking = ChatAnthropic(
    model='claude-opus-4-6',
    api_key=ANTHROPIC_API_KEY,
    max_tokens=8000,
    thinking={'type': 'enabled', 'budget_tokens': 3000},
)

FUTURE_RISK_PROMPT = """
You are the Future Risk Agent for a rental property evaluator.

Assess long-term risk factors that could affect the desirability or liveability of this property over a 2-5 year horizon.

Research:
1. Neighborhood gentrification or decline trajectory (recent permit applications, new businesses, crime trend)
2. Rent control laws in the city/county — tenant protections
3. Developer activity: large construction projects nearby that could affect noise/parking/traffic
4. School district trajectory (if family profile)
5. Climate risk trajectory: is flood/fire/heat risk increasing?
6. City budget health: municipal services cuts, police/fire staffing
7. Landlord history: any pattern of evictions, code violations, or lawsuits

Output ONLY valid JSON:
{"score": <integer 0-100>, "summary": "<one sentence>", "details": "<3-5 paragraphs>"}
"""

from app.tools.search import web_search as ws_tool
agent_thinking = create_react_agent(llm_thinking, [ws_tool], prompt=FUTURE_RISK_PROMPT)

result_thinking = await agent_thinking.ainvoke({
    'messages': [('user', f'Evaluate future risk for: {TEST_ADDRESS} (zip: {TEST_ZIP})\nProfile: {TEST_PROFILE}')]
})

final = next(
    (m.content for m in reversed(result_thinking['messages'])
     if hasattr(m, 'content') and isinstance(m.content, str) and m.content.strip()),
    ''
)
print(final)

## 7. Prompt refinement checklist

After running each agent, check:

- [ ] **Did it use tools?** Check `[ToolMessage]` blocks in the trace. If not, the prompt isn't triggering tool use.
- [ ] **Is the score calibrated?** 50 = fallback/error. Real scores should vary 30-90.
- [ ] **Is the summary one sentence?** If it's multiple sentences, add enforcement to the prompt.
- [ ] **Does the details field reference the user's profile?** ("for your vegetarian diet...", "as a no-car commuter...")
- [ ] **Does the JSON parse cleanly?** If not, add `Respond ONLY with JSON. No text before or after.`
- [ ] **Did it time out?** If so, reduce the number of tool calls or simplify the prompt scope.

When a prompt passes all checks → copy it into the corresponding `backend/app/agents/<agent>.py` file.